# Ingestion coverage — code-review-graph on `guava`

Google Guava (Java core libraries, large: ~3,270 `.java` files, commit fce80fd, shallow clone).
Fills the team's `Ingestion_Coverage.xlsx` for this project. **Functions expected** come from javac's own
parser (`ref_java.py` → `java-ref/ListMethods.java`). Scale test: the build and the javac step take longer.

Every run is a new iteration in `results/guava/runs/iter-NNN_<date>_<time>/` and adds lines to
`results/guava/runs.csv`; nothing from an earlier run is overwritten.

Run: `./run.sh guava` (from `notebooks/`), or Run All here and Save As into the iteration folder.

In [1]:
APP = 'guava'   # this notebook's project (folder under apps/)
RUN = ''        # iteration folder name; empty = next free iter-NNN (run.sh sets it)

In [2]:
import sys; sys.path.insert(0, '.')
import pandas as pd
import coverage_lab as C
import ref_java
REFERENCES = [ref_java]   # this project's languages that get a functions-expected count
pd.set_option('display.max_rows', 300); pd.set_option('display.max_colwidth', 120)
RUN = RUN or C.next_run_id(APP)
OUT = C.run_dir(APP, RUN)   # results/<APP>/runs/<RUN>/
stamp = C.run_stamp(APP)
{**stamp, 'run': RUN}

$ git rev-parse --short HEAD 
$ /Users/thiru/MySpace/Celsior/repo/code-graph-rnd/.venv-crg240/bin/code-review-graph --version 


{'app': 'guava',
 'app_commit': 'fce80fd',
 'tool': 'code-review-graph 2.4.0',
 'run_at': '2026-11-17T22:47:39',
 'run': 'iter-003_2026-11-17_2247'}

## 1. Build the graph exactly as Synthetix does
`CRG_DATA_DIR=<dir> code-review-graph build --repo <root> -q`, after removing the repo from the tool's registry.

In [3]:
db = C.build(APP, OUT)
db

$ /Users/thiru/MySpace/Celsior/repo/code-graph-rnd/.venv-crg240/bin/code-review-graph repos 
$ /Users/thiru/MySpace/Celsior/repo/code-graph-rnd/.venv-crg240/bin/code-review-graph build --repo /Users/thiru/MySpace/Celsior/repo/code-graph-rnd/apps/guava -q (env {'CRG_DATA_DIR': '/Users/thiru/MySpace/Celsior/repo/code-graph-rnd/results/guava/runs/iter-003_2026-11-17_2247/crg'})


PosixPath('/Users/thiru/MySpace/Celsior/repo/code-graph-rnd/results/guava/runs/iter-003_2026-11-17_2247/crg/graph.db')

## 2. Sheet 1 — one row per code file
Read? / Why come from the tool's own filter and a re-parse with the same grammar.

In [4]:
rows, funcs, left_out = C.coverage(APP, db, REFERENCES)
sheet1 = pd.DataFrame([{k: v for k, v in r.__dict__.items() if k != 'expected'} for r in rows])
sheet1.read.value_counts()

$ java ListMethods.java guava (3267 files on stdin)


read
Fully read     3270
Partly read      14
Name: count, dtype: int64

### Files left out of sheet 1 (not code)
No language in the tool and not a programming-language extension: docs, licences, images, Dockerfiles, config.

In [5]:
print(len(left_out), 'files left out')
pd.Series([p.rsplit('.', 1)[-1] if '.' in p.rsplit('/', 1)[-1] else '(no extension)' for p in left_out]).value_counts()

70 files left out


xml               33
txt               13
pro                6
md                 5
(no extension)     5
gitignore          2
properties         2
gitattributes      1
jar                1
bat                1
cmd                1
Name: count, dtype: int64

In [6]:
sheet1[sheet1.read != 'Fully read'][['file', 'language', 'lines', 'read', 'why', 'lines_read', 'notes']]

,file,language,lines,read,why,lines_read,notes
279,android/guava-testlib/src/com/google/common/testing/EqualsTester.java,java,167,Partly read,Parse error,166,1 lines inside syntax errors: 111
1020,android/guava/src/com/google/common/base/Objects.java,java,82,Partly read,Parse error,81,1 lines inside syntax errors: 77
1025,android/guava/src/com/google/common/base/Platform.java,java,119,Partly read,Parse error,118,1 lines inside syntax errors: 82
1026,android/guava/src/com/google/common/base/Preconditions.java,java,1540,Partly read,Parse error,1534,"6 lines inside syntax errors: 161-162, 564-565, 998-999"
1035,android/guava/src/com/google/common/base/Strings.java,java,319,Partly read,Parse error,318,1 lines inside syntax errors: 266
1041,android/guava/src/com/google/common/base/Verify.java,java,540,Partly read,Parse error,536,"4 lines inside syntax errors: 121-122, 528-529"
1601,guava-gwt/src-super/com/google/common/base/super/com/google/common/base/Platform.java,java,74,Partly read,Parse error,73,1 lines inside syntax errors: 65
1936,guava-testlib/src/com/google/common/testing/EqualsTester.java,java,167,Partly read,Parse error,166,1 lines inside syntax errors: 111
2699,guava/src/com/google/common/base/Objects.java,java,82,Partly read,Parse error,81,1 lines inside syntax errors: 77
2704,guava/src/com/google/common/base/Platform.java,java,115,Partly read,Parse error,114,1 lines inside syntax errors: 83


## 3. Functions: expected (language's own parser) vs found (graph.db)

In [7]:
matched = C.match_functions(rows, funcs)
matched.groupby(['language', 'status']).size()

language  status   
java      found        59188
          not found       48
dtype: int64

In [8]:
matched[matched.status != 'found']

,language,file,function,line,status
920,java,android/guava-testlib/src/com/google/common/collect/testing/features/CollectionFeature.java,value,122,not found
921,java,android/guava-testlib/src/com/google/common/collect/testing/features/CollectionFeature.java,absent,124,not found
926,java,android/guava-testlib/src/com/google/common/collect/testing/features/CollectionSize.java,value,94,not found
927,java,android/guava-testlib/src/com/google/common/collect/testing/features/CollectionSize.java,absent,96,not found
948,java,android/guava-testlib/src/com/google/common/collect/testing/features/ListFeature.java,value,64,not found
949,java,android/guava-testlib/src/com/google/common/collect/testing/features/ListFeature.java,absent,66,not found
952,java,android/guava-testlib/src/com/google/common/collect/testing/features/MapFeature.java,value,91,not found
953,java,android/guava-testlib/src/com/google/common/collect/testing/features/MapFeature.java,absent,93,not found
956,java,android/guava-testlib/src/com/google/common/collect/testing/features/SetFeature.java,value,52,not found
957,java,android/guava-testlib/src/com/google/common/collect/testing/features/SetFeature.java,absent,54,not found


### Files the reference parser could not fully check
Listed, never silently dropped (a syntax error, or code the reference cannot parse).

In [9]:
C.could_not_check(rows)

,language,file,reason


### Unnamed functions (callbacks, lambdas) — not counted as expected
Shown per file with the lines they cover. Their code is still walked by the tool; only a separate row is missing.

In [10]:
cb = C.callbacks_table(rows)
print(cb.callbacks.sum(), 'unnamed functions,', cb.lines.sum(), 'lines')
cb.sort_values('callbacks', ascending=False)

6153 unnamed functions, 10806 lines


,language,file,callbacks,lines
326,java,android/guava-tests/test/com/google/common/util/concurrent/FuturesTest.java,161,367
759,java,guava-tests/test/com/google/common/util/concurrent/FuturesTest.java,161,367
608,java,guava-tests/test/com/google/common/collect/MapsTest.java,98,98
302,java,android/guava-tests/test/com/google/common/util/concurrent/AbstractClosingFutureTest.java,84,365
735,java,guava-tests/test/com/google/common/util/concurrent/AbstractClosingFutureTest.java,84,365
...,...,...,...,...
247,java,android/guava-tests/test/com/google/common/io/PatternFilenameFilterTest.java,1,1
245,java,android/guava-tests/test/com/google/common/io/LineBufferTest.java,1,1
244,java,android/guava-tests/test/com/google/common/io/FlushablesTest.java,1,1
241,java,android/guava-tests/test/com/google/common/io/CountingOutputStreamTest.java,1,1


## 4. Sheet 2 — per language

In [11]:
sheet2 = C.parsers_table(APP, rows, matched, RUN, REFERENCES)
sheet2

,Language,Package,Repo,Files tried,Files parsed,Functions expected,Functions found,Evidence,Notes
0,bash,code-review-graph 2.4.0 (grammar: bash),guava,5,5,NaN,NaN,results/guava/runs/iter-003_2026-11-17_2247/coverage.ipynb,no reference parser for this language yet
1,html,code-review-graph 2.4.0 (grammar: html),guava,1,1,NaN,NaN,results/guava/runs/iter-003_2026-11-17_2247/coverage.ipynb,no reference parser for this language yet
2,java,code-review-graph 2.4.0 (grammar: java),guava,3267,3267,59236.0,59188.0,results/guava/runs/iter-003_2026-11-17_2247/coverage.ipynb,"expected from javac parser (JDK, java-ref); lost to same-name overwrite: 0; not found: 48; unnamed functions not cou..."
3,javascript,code-review-graph 2.4.0 (grammar: javascript),guava,2,2,NaN,NaN,results/guava/runs/iter-003_2026-11-17_2247/coverage.ipynb,no reference parser for this language yet
4,json,code-review-graph 2.4.0 (grammar: json),guava,1,1,NaN,NaN,results/guava/runs/iter-003_2026-11-17_2247/coverage.ipynb,no reference parser for this language yet
5,kotlin,code-review-graph 2.4.0 (grammar: kotlin),guava,2,2,NaN,NaN,results/guava/runs/iter-003_2026-11-17_2247/coverage.ipynb,no reference parser for this language yet
6,yaml,code-review-graph 2.4.0 (grammar: yaml),guava,6,6,NaN,NaN,results/guava/runs/iter-003_2026-11-17_2247/coverage.ipynb,no reference parser for this language yet


## 5. Sheet 3 — summary (same formulas as the workbook)

In [12]:
s = sheet1
summary = {'Files': len(s), 'Fully read': (s.read == 'Fully read').sum(), 'Partly read': (s.read == 'Partly read').sum(),
  'Not read': (s.read == 'Not read').sum(), 'Lines': int(s.lines.fillna(0).sum()), 'Lines read': int(s.lines_read.fillna(0).sum()),
  'Not shown to user': ((s.read != 'Fully read') & (s.shown == 'No')).sum()}
summary['% lines read'] = round(100 * summary['Lines read'] / summary['Lines'], 1)
summary

{'Files': 3284,
 'Fully read': np.int64(3270),
 'Partly read': np.int64(14),
 'Not read': np.int64(0),
 'Lines': 803591,
 'Lines read': 803561,
 'Not shown to user': np.int64(14),
 '% lines read': 100.0}

In [13]:
s[s.read == 'Not read'].why.value_counts()

Series([], Name: count, dtype: int64)

## 6. Write the workbook

In [14]:
C.write_workbook(APP, rows, sheet2, OUT)

/Users/thiru/MySpace/Celsior/repo/code-graph-rnd/.venv-crg240/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


PosixPath('/Users/thiru/MySpace/Celsior/repo/code-graph-rnd/results/guava/runs/iter-003_2026-11-17_2247/Ingestion_Coverage_guava.xlsx')

## 7. Add this run to the repo's run log
`results/<repo>/runs.csv`, append-only: one line per run (per language with a reference parser).

In [15]:
log = C.log_run(APP, RUN, stamp, summary, sheet2)
pd.read_csv(log)

,run,run_at,app_commit,tool,files,fully_read,partly_read,not_read,lines,lines_read,pct_lines_read,language,functions_expected,functions_found,notes
0,iter-001_2026-11-17_2018,2026-11-17T20:18:30,fce80fd,code-review-graph 2.3.8,3282,3260,14,8,803297,802389,99.9,bash,NaN,NaN,no reference parser for this language yet
1,iter-001_2026-11-17_2018,2026-11-17T20:18:30,fce80fd,code-review-graph 2.3.8,3282,3260,14,8,803297,802389,99.9,java,59236.0,52755.0,NaN
2,iter-001_2026-11-17_2018,2026-11-17T20:18:30,fce80fd,code-review-graph 2.3.8,3282,3260,14,8,803297,802389,99.9,javascript,NaN,NaN,no reference parser for this language yet
3,iter-001_2026-11-17_2018,2026-11-17T20:18:30,fce80fd,code-review-graph 2.3.8,3282,3260,14,8,803297,802389,99.9,yaml,NaN,NaN,no reference parser for this language yet
4,iter-002_2026-11-17_2243,2026-11-17T22:43:33,fce80fd,code-review-graph 2.3.8,3282,3260,14,8,803297,802389,99.9,bash,NaN,NaN,no reference parser for this language yet
5,iter-002_2026-11-17_2243,2026-11-17T22:43:33,fce80fd,code-review-graph 2.3.8,3282,3260,14,8,803297,802389,99.9,java,59236.0,52755.0,NaN
6,iter-002_2026-11-17_2243,2026-11-17T22:43:33,fce80fd,code-review-graph 2.3.8,3282,3260,14,8,803297,802389,99.9,javascript,NaN,NaN,no reference parser for this language yet
7,iter-002_2026-11-17_2243,2026-11-17T22:43:33,fce80fd,code-review-graph 2.3.8,3282,3260,14,8,803297,802389,99.9,yaml,NaN,NaN,no reference parser for this language yet
8,iter-003_2026-11-17_2247,2026-11-17T22:47:39,fce80fd,code-review-graph 2.4.0,3284,3270,14,0,803591,803561,100.0,bash,NaN,NaN,no reference parser for this language yet
9,iter-003_2026-11-17_2247,2026-11-17T22:47:39,fce80fd,code-review-graph 2.4.0,3284,3270,14,0,803591,803561,100.0,html,NaN,NaN,no reference parser for this language yet
